<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 생성 모델 · 06. CycleGAN

## Unpaired Image-to-Image Translation Using Cycle-Consistent Adversarial Networks

- **원 논문:** [Unpaired Image-to-Image Translation Using Cycle-Consistent Adversarial Networks](https://openaccess.thecvf.com/content_iccv_2017/html/Zhu_Unpaired_Image-To-Image_Translation_ICCV_2017_paper.html)
- **저자 / 발표:** Jun-Yan Zhu, Taesung Park, Phillip Isola, and Alexei A. Efros · ICCV (2017)
- **난이도 / 예상 시간:** 중급 · 약 65분
- **선수 지식:** GAN, cycle consistency, two-domain translation
- **로컬 학습 데이터:** `data/field_curriculum/generative_samples.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** unpaired 2D domains에서 두 generator와 두 discriminator를 adversarial+cycle loss로 학습한다.

**축소하거나 생략한 부분:** 논문은 unpaired image collections와 ResNet generators를 사용한다. 축소판은 샘플 pairing을 섞고 양방향 mapping/cycle을 보존한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.1, Eq. (1) | adversarial domain mapping | G:A→B와 D_B의 adversarial objective를 구성한다. |
| §3.2, Eq. (2) | cycle consistency | F(G(a))≈a 및 G(F(b))≈b의 L1 loss를 구현한다. |
| §3.3, Eq. (3)–(4) | full objective | 두 adversarial loss와 lambda-weighted cycle loss를 공동 최적화한다. |

## 핵심 재현

        Eq. (2)는 직접 짝이 없는 상황에서 $F(G(x))\approx x$, $G(F(y))\approx y$를 요구합니다.
        B mini-batch를 별도로 섞어 index pairing을 제거하고 두 방향을 동시에 학습합니다.

In [ ]:
from pathlib import Path
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(361)
np.random.seed(361)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# These teaching batches are intentionally tiny; GPU launch/transfer
# overhead can dominate, so use AI_LAB_DEVICE=cpu for minimum wall time.

DATA_PATH = Path("data/field_curriculum/generative_samples.npz")
assert DATA_PATH.exists(), f"준비된 로컬 데이터가 없습니다: {DATA_PATH}"
raw = np.load(DATA_PATH)
expected = {"points", "point_labels", "images", "image_labels", "domain_a", "domain_b"}
assert expected.issubset(raw.files), f"NPZ 키 불일치: {sorted(raw.files)}"
points = torch.tensor(raw["points"], dtype=torch.float32)
point_labels = torch.tensor(raw["point_labels"], dtype=torch.long)
images = torch.tensor(raw["images"], dtype=torch.float32)
image_labels = torch.tensor(raw["image_labels"], dtype=torch.long)
domain_a = torch.tensor(raw["domain_a"], dtype=torch.float32)
domain_b = torch.tensor(raw["domain_b"], dtype=torch.float32)
points, point_labels, images, image_labels, domain_a, domain_b = ACCELERATOR.move(
    points, point_labels, images, image_labels, domain_a, domain_b
)


# Seeded random draws stay on CPU so CUDA, MPS, and CPU follow the same
# sampling stream; complete tensors are then transferred to the lab device.
def randn_device(size, *, generator):
    return torch.randn(size, generator=generator).to(DEVICE)


def rand_device(size, *, generator):
    return torch.rand(size, generator=generator).to(DEVICE)


def randint_device(high, size, *, generator):
    return torch.randint(high, size, generator=generator).to(DEVICE)


if float(images.min()) < 0.0:
    images_01 = ((images + 1.0) / 2.0).clamp(0.0, 1.0)
else:
    images_01 = images.clamp(0.0, 1.0)
assert points.shape == (600, 2) and images.shape == (300, 1, 8, 8)
assert domain_a.shape == domain_b.shape == (600, 2)
print(ACCELERATOR.summary())
print("local data:", tuple(points.shape), tuple(images.shape), tuple(domain_a.shape))

## 포트폴리오 구현 설계: 수식 → 책임 → 검증

논문의 핵심 목적을 실행 가능한 객체의 `forward`, `loss`, `update`, `evaluate`로
나눕니다. 아래 식이 클래스 내부에서 생략되지 않고 어떤 tensor 연산이 되는지
메서드별로 추적해 보세요.

$$
\mathcal L_{cyc}=\mathbb E_x\lVert F(G(x))-x\rVert_1+\mathbb E_y\lVert G(F(y))-y\rVert_1
$$

- **기호와 역할:** $G:A\to B$와 $F:B\to A$는 두 generator, $D_A/D_B$는 각 domain critic입니다.
- **shape 계약:** `unpaired A/B [N, 2] ↔ mapped/cycled [N, 2]`
- **논문 위치:** `§3.1, Eq. (1)`의 **adversarial domain mapping**
- **코드 Task:** 양방향 변환, cycle consistency, 두 adversarial loss의 교대 update를 구현합니다.
- **학습·평가 흐름:** 독립 A/B batch → 두 D update → 두 G+cycle update → cycle L1 평가
- **원문 대비 한계:** 논문은 unpaired image collections와 ResNet generators를 사용한다. 축소판은 샘플 pairing을 섞고 양방향 mapping/cycle을 보존한다.
- **구현 이유:** 수식의 tensor 경계와 optimizer 책임을 클래스 메서드별로
  분리해 코드와 논문을 한 줄씩 대조할 수 있게 합니다.
- **완료 증거:** 마지막 셀에서 `update`, parameter 변화, 평가 metric을
  모두 `assert`합니다.

정답 클래스는 교육용 편의 함수가 핵심 계산을 대신하지 않도록 loss와 update 순서를
메서드 본문에 드러냅니다. 실습본에서는 같은 공개 API를 유지한 채 TODO를 채우세요.

In [ ]:
class CycleConsistentTranslationLab(nn.Module):
    """CycleGAN의 핵심 학습·평가 경로. 입출력 계약: unpaired A/B [N, 2] ↔ mapped/cycled [N, 2]."""

    def __init__(
        self,
        generator_ab,
        generator_ba,
        discriminator_a,
        discriminator_b,
        generator_optimizer,
        discriminator_optimizer,
    ):
        """구성 요소와 optimizer를 저장한다. 학습은 update에서만 수행한다."""
        super().__init__()
        self.generator_ab = generator_ab
        self.generator_ba = generator_ba
        self.discriminator_a = discriminator_a
        self.discriminator_b = discriminator_b
        self.generator_optimizer = generator_optimizer
        self.discriminator_optimizer = discriminator_optimizer
        self.cycle_weight = 8.0

    def config(self):
        """재현 범위와 핵심 하이퍼파라미터를 dict로 반환한다."""
        return {"domain_dim": 2, "cycle_weight": self.cycle_weight}

    def forward(self, domain_a_batch, domain_b_batch):
        """논문 순전파를 계산한다. shape 계약: unpaired A/B [N, 2] ↔ mapped/cycled [N, 2]."""
        fake_b = self.generator_ab(domain_a_batch)
        fake_a = self.generator_ba(domain_b_batch)
        cycle_a = self.generator_ba(fake_b)
        cycle_b = self.generator_ab(fake_a)
        return (fake_a, fake_b, cycle_a, cycle_b)

    def loss(self, domain_a_batch, domain_b_batch):
        """논문의 목적 함수를 미분 가능한 scalar tensor로 반환한다."""
        fake_a, fake_b, cycle_a, cycle_b = self.forward(domain_a_batch, domain_b_batch)
        cycle = F.l1_loss(cycle_a, domain_a_batch)
        cycle = cycle + F.l1_loss(cycle_b, domain_b_batch)
        adversarial = F.binary_cross_entropy_with_logits(
            self.discriminator_b(fake_b), torch.ones_like(self.discriminator_b(fake_b))
        )
        adversarial = adversarial + F.binary_cross_entropy_with_logits(
            self.discriminator_a(fake_a), torch.ones_like(self.discriminator_a(fake_a))
        )
        return (adversarial + self.cycle_weight * cycle, cycle)

    def update(self, domain_a_batch, domain_b_batch):
        """zero_grad, backward, step을 수행하고 유한한 float loss를 반환한다."""
        with torch.no_grad():
            detached_fake_a = self.generator_ba(domain_b_batch)
            detached_fake_b = self.generator_ab(domain_a_batch)
        discriminator_loss = F.binary_cross_entropy_with_logits(
            self.discriminator_a(domain_a_batch),
            torch.ones_like(self.discriminator_a(domain_a_batch)),
        )
        discriminator_loss = discriminator_loss + F.binary_cross_entropy_with_logits(
            self.discriminator_a(detached_fake_a),
            torch.zeros_like(self.discriminator_a(detached_fake_a)),
        )
        discriminator_loss = discriminator_loss + F.binary_cross_entropy_with_logits(
            self.discriminator_b(domain_b_batch),
            torch.ones_like(self.discriminator_b(domain_b_batch)),
        )
        discriminator_loss = discriminator_loss + F.binary_cross_entropy_with_logits(
            self.discriminator_b(detached_fake_b),
            torch.zeros_like(self.discriminator_b(detached_fake_b)),
        )
        self.discriminator_optimizer.zero_grad()
        discriminator_loss.backward()
        self.discriminator_optimizer.step()
        generator_loss, cycle = self.loss(domain_a_batch, domain_b_batch)
        self.generator_optimizer.zero_grad()
        generator_loss.backward()
        self.generator_optimizer.step()
        return (
            float(discriminator_loss.detach()),
            float(generator_loss.detach()),
            float(cycle.detach()),
        )

    def evaluate(self, domain_a_batch, domain_b_batch):
        """no_grad 경로에서 논문과 연결된 정량 지표 dict를 반환한다."""
        with torch.no_grad():
            fake_a, fake_b, cycle_a, cycle_b = self.forward(
                domain_a_batch, domain_b_batch
            )
            cycle = F.l1_loss(cycle_a, domain_a_batch)
            cycle = cycle + F.l1_loss(cycle_b, domain_b_batch)
        return {
            "cycle_l1": float(cycle),
            "fake_a_shape": tuple(fake_a.shape),
            "fake_b_shape": tuple(fake_b.shape),
        }

### 단계 2. 핵심 연산 검증

- **논문의 어느 부분인가:** `§3.2, Eq. (2)` — **cycle consistency**
- **구현 이유:** F(G(a))≈a 및 G(F(b))≈b의 L1 loss를 구현한다.
- **읽을 코드:** 아래 `implementation` 셀에서 중간 tensor의 shape, gradient가 흐르는
  경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
def mapper():
    return nn.Sequential(nn.Linear(2, 32), nn.Tanh(), nn.Linear(32, 2))


def disc():
    return nn.Sequential(nn.Linear(2, 32), nn.LeakyReLU(0.2), nn.Linear(32, 1))


G_AB, G_BA, D_A, D_B = ACCELERATOR.move(mapper(), mapper(), disc(), disc())


def cycle_loss(a, b):
    return F.l1_loss(G_BA(G_AB(a)), a) + F.l1_loss(G_AB(G_BA(b)), b)


assert cycle_loss(domain_a[:4], domain_b[:4]).ndim == 0

### 단계 3. 학습·업데이트

- **논문의 어느 부분인가:** `§3.3, Eq. (3)–(4)` — **full objective**
- **구현 이유:** 두 adversarial loss와 lambda-weighted cycle loss를 공동 최적화한다.
- **읽을 코드:** 아래 `training` 셀에서 중간 tensor의 shape, gradient가 흐르는
  경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
opt_d = torch.optim.Adam(list(D_A.parameters()) + list(D_B.parameters()), lr=0.003)
opt_g = torch.optim.Adam(list(G_AB.parameters()) + list(G_BA.parameters()), lr=0.004)
rng = torch.Generator().manual_seed(362)
history = []
with torch.no_grad():
    initial_cycle = float(cycle_loss(domain_a, domain_b))

portfolio_model = CycleConsistentTranslationLab(
    G_AB,
    G_BA,
    D_A,
    D_B,
    opt_g,
    opt_d,
)
initial_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in G_AB.parameters()]
)
for _ in range(220):
    index_a = randint_device(len(domain_a), (96,), generator=rng)
    index_b = randint_device(len(domain_b), (96,), generator=rng)
    history.append(portfolio_model.update(domain_a[index_a], domain_b[index_b]))
final_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in G_AB.parameters()]
)
parameter_delta = float((final_parameters - initial_parameters).abs().sum())
assert parameter_delta > 0.0 and np.isfinite(history).all()

### 단계 4. 평가·시각화

        - **논문의 어느 부분인가:** `§3.2, Eq. (2)` — **cycle consistency**
        - **구현 이유:** F(G(a))≈a 및 G(F(b))≈b의 L1 loss를 구현한다.
        - **읽을 코드:** 아래 `evaluation` 셀에서 중간 tensor의 shape, gradient가 흐르는
          경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

        실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
        주석으로 적은 뒤 실습하세요.
        - **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

- **평가 범위:** 이 값은 작은 학습 batch의 메커니즘 진단이며, held-out 생성 품질이나 원 논문의 benchmark 성능으로 해석하지 않습니다.

In [ ]:
portfolio_metrics = portfolio_model.evaluate(domain_a, domain_b)
assert portfolio_metrics

with torch.no_grad():
    mapped = G_AB(domain_a)
    final_cycle = float(cycle_loss(domain_a, domain_b))
print(f"cycle L1: {initial_cycle:.4f} -> {final_cycle:.4f}")
assert final_cycle < initial_cycle
a_cpu, mapped_cpu, b_cpu = (
    domain_a.detach().cpu(),
    mapped.detach().cpu(),
    domain_b.detach().cpu(),
)
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].scatter(a_cpu[:, 0], a_cpu[:, 1], s=7)
axes[0].set_title("A (unpaired)")
axes[1].scatter(mapped_cpu[:, 0], mapped_cpu[:, 1], s=7)
axes[1].set_title("G_AB(A)")
axes[2].scatter(b_cpu[:, 0], b_cpu[:, 1], s=7)
axes[2].set_title("B")
fig.tight_layout()
plt.show()

서로 역인 임의 변환도 cycle loss는 0이 될 수 있습니다. adversarial loss가 각 방향의 출력 분포를 실제 target domain에 맞춥니다.

### 단계 5. 통합 클래스가 실제 update와 evaluate를 소유하는지 검증

- **논문 위치:** `§3.1, Eq. (1)`의 **adversarial domain mapping**
- **핵심 식:**

  $$
  \mathcal L_{cyc}=\mathbb E_x\lVert F(G(x))-x\rVert_1+\mathbb E_y\lVert G(F(y))-y\rVert_1
  $$

- **입출력 shape:** `unpaired A/B [N, 2] ↔ mapped/cycled [N, 2]`
- **구현 이유:** 앞 셀은 수식을 작은 연산으로 분해해 확인하고,
  이 셀은 같은 구성 요소를 `CycleConsistentTranslationLab`에 주입해
  최종 학습·평가 경로를 하나로 묶습니다. G:A→B와 D_B의 adversarial objective를 구성한다.
- **완료 증거:** `update(...)`가 유한한 loss를 반환하고,
  `parameter_delta > 0`이며, `evaluate(...)`의 논문 대응 지표가 유한해야 합니다.
  `config()`만 호출하는 것은 실행 재현으로 보지 않습니다.

In [ ]:
config_contract = portfolio_model.config()
numeric_metrics = [
    float(value)
    for value in portfolio_metrics.values()
    if isinstance(value, (int, float))
]
assert config_contract and len(history) > 0
assert np.isfinite(np.asarray(history, dtype=float)).all()
assert parameter_delta > 0.0
assert numeric_metrics and np.isfinite(numeric_metrics).all()
print(
    {
        "class": type(portfolio_model).__name__,
        "steps": len(history),
        "parameter_delta": parameter_delta,
        "metrics": portfolio_metrics,
    }
)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 논문은 unpaired image collections와 ResNet generators를 사용한다. 축소판은 샘플 pairing을 섞고 양방향 mapping/cycle을 보존한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.